In [3]:
%pip install -qU langchain langchain-community langchain-text-splitters pypdf nltk langchain-google-genai langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 779.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/27

In [6]:
import os
import urllib.request
import nltk
import time
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import NLTKTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Environment variables
os.environ["USER_AGENT"] = "MyRAGApp/1.0 (legal-analyzer)"
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# 2. Download NLTK tokenizers
nltk.download("punkt")
nltk.download("punkt_tab")

# 3. Path to PDF contract
pdf_path = "/content/nasdaq_master_services_agreement.pdf"

if not os.path.exists(pdf_path):
    print("Downloading official Nasdaq Master Services Agreement PDF...")
    url = "https://listingcenter.nasdaq.com/assets/Master%20Services%20Agreement%20Form.pdf"
    urllib.request.urlretrieve(url, pdf_path)
    print("✓ Real-world PDF successfully downloaded!")

# 4. Load the PDF Document
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 5. Split document into larger chunks (1000 characters) to reduce total request count
text_splitter = NLTKTextSplitter(chunk_size=1000, chunk_overlap=100)
tokens_chunks = text_splitter.split_documents(docs)

# 6. Embed and store in Chroma Vector Database with conservative rate-limiting
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
save_to_dir = "/content/legal_chroma_db"

vector_db = Chroma(persist_directory=save_to_dir, embedding_function=embeddings)

print(f"Total chunks to process: {len(tokens_chunks)}")

# Process in smaller batches of 15 with 60-second cooldowns
batch_size = 15
for i in range(0, len(tokens_chunks), batch_size):
    batch = tokens_chunks[i : i + batch_size]

    vector_db.add_documents(batch)
    print(f"✓ Successfully indexed chunks {i + 1} to {min(i + batch_size, len(tokens_chunks))}")

    # Pause 60s if more chunks remain to reset the 100 RPM window
    if i + batch_size < len(tokens_chunks):
        print("Waiting 60 seconds for API quota reset...")
        time.sleep(60)

# 7. Define Legal Analysis Prompt Template
qna_template = "\n".join([
    "You are an expert Legal Risk & Contract Analysis Assistant.",
    "Your objective is to analyze contract clauses provided in the context and evaluate legal liabilities.",
    "",
    "Instructions:",
    "1. Answer the user's question based strictly on the provided contract context.",
    "2. If applicable, identify the Risk Level (HIGH, MEDIUM, LOW) of the found provisions.",
    "3. Highlight potential liabilities, limitations, or restrictions for the parties involved.",
    "4. Suggest a concrete 'Redline / Renegotiation Recommendation' if the clause poses high risk.",
    "5. If the provision or answer is not in the context, explicitly state: 'NO MATCHING CLAUSE FOUND IN CONTRACT'.",
    "",
    "### Context:",
    "{context}",
    "",
    "### Legal Query / Clause to Analyze:",
    "{question}",
    "",
    "### Analysis & Redline Report:",
])

qna_prompt = PromptTemplate.from_template(qna_template)
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

# 8. Create LCEL QA Chain
qa_chain = qna_prompt | llm | StrOutputParser()

print("\n✓ Legal Contract Analyzer RAG Pipeline ready!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total chunks to process: 66
✓ Successfully indexed chunks 1 to 15
Waiting 60 seconds for API quota reset...
✓ Successfully indexed chunks 16 to 30
Waiting 60 seconds for API quota reset...
✓ Successfully indexed chunks 31 to 45
Waiting 60 seconds for API quota reset...
✓ Successfully indexed chunks 46 to 60
Waiting 60 seconds for API quota reset...
✓ Successfully indexed chunks 61 to 66

✓ Legal Contract Analyzer RAG Pipeline ready!


In [11]:
# @title Legal Contract Analysis & Risk Review
# @markdown Select or enter a legal query to analyze clauses in the Nasdaq Agreement:

legal_query = "What is the definition of Confidential Information and what are its exceptions?" # @param ["What is the definition of Confidential Information and what are its exceptions?", "What are the limitations on usage and license restrictions?", "What are the rules regarding data protection and permissions?", "Custom Query..."]
custom_query = "" # @param {type:"string"}

# Use custom query if provided
user_question = custom_query.strip() if legal_query == "Custom Query..." and custom_query.strip() else legal_query

# --- RAG Execution ---
if user_question.strip():
    # 1. Retrieve relevant contract chunks from vector database
    similar_docs = vector_db.similarity_search(user_question, k=4)
    context_text = "\n\n".join(doc.page_content for doc in similar_docs)

    # 2. Invoke QA chain
    analysis = qa_chain.invoke({
        "context": context_text,
        "question": user_question
    })

    # 3. Print analysis report
    print(f"==================================================")
    print(f" QUERY: {user_question}")
    print(f"==================================================\n")
    print(analysis)
else:
    print("Please select or enter a valid legal query.")

 QUERY: What is the definition of Confidential Information and what are its exceptions?

### **Analysis & Redline Report**

#### **1. Answer to Legal Query**
**Definition of Confidential Information:**  
"Confidential Information" includes all information, technology, data, and other materials disclosed or made available by the Disclosing Party (directly or indirectly) to the Receiving Party, regardless of the method of disclosure and regardless of whether it is marked or identified as "confidential."

**Exceptions:**  
Confidential Information does **not** include any information, technology, data, or materials that:
1. **Public Domain:** Are or become generally available to or known by the public, other than through a breach of the agreement by the Receiving Party.
2. **Third-Party Disclosure:** Are disclosed to the Receiving Party on a non-confidential basis by a third party, provided that (to the Receiving Party’s knowledge) the third party is not legally or contractually prohibite